# 03 - Data Cleaning and Geography

**Input:** the original archive export (`data/processed/aotd_articles_enriched.csv`)
**Output:** none - this notebook explains and checks the cleaning. The analytics
table is owned by the pipeline (see the last section).

This is where the messiest column in the dataset gets fixed.

`label_location_raw` is free text that a label owner typed into their own
Bandcamp profile. Across ~2,300 articles it contains every failure mode you'd
expect and several you wouldn't. Left alone it produces **422 distinct
"locations"** — many of which are the same place spelled differently — and it
is unusable for a map.

After cleaning: **407 distinct locations, 80 countries, and zero unresolved
values.** (Those are the original export's numbers; each refresh adds a few
more raw spellings for the same rules to absorb.)

In [1]:
import sys
from pathlib import Path

# Make the pipeline package importable without installing it, so the notebook
# runs on a fresh clone. `pip install -e .` also works and is preferred.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from bandcamp_aotd.logging_config import setup_logging

setup_logging()

In [2]:
import pandas as pd
from bandcamp_aotd import config
from bandcamp_aotd.legacy import load_legacy_csv

df = load_legacy_csv(config.ENRICHED_CSV)
print(f"{df['label_location_raw'].nunique()} distinct raw location strings")
df["label_location_raw"].dropna().sample(10, random_state=7).tolist()

14:07:57 | INFO    | bandcamp_aotd.legacy           | Loaded 2288 rows x 25 columns from aotd_articles_enriched.csv


422 distinct raw location strings


['Paris, France',
 'New York, New York',
 'London, UK',
 'Finland',
 'Malmö, Sweden',
 'New York, New York',
 'Los Angeles, California',
 'Richmond, Virginia',
 'New York, New York',
 'Auckland, New Zealand']

## The problems, and the order they have to be fixed in

The cleaner is an explicit nine-step pipeline in
`src/bandcamp_aotd/transform/locations.py`. **Order is load-bearing**, and two
steps in particular have to happen when they do:

| # | Step | Fixes |
|---|---|---|
| 1 | `fix_whitespace` | `"Osaka ,  Japan"` -> `"Osaka, Japan"` |
| 2 | `fix_cjk` | CJK placenames -> romanised |
| 3 | `fix_dc` | six spellings of Washington D.C. -> one |
| 4 | `fix_turkish_dotted_i` | U+0130 -> plain I |
| 5 | `fix_case` | title-case everything |
| 6 | `fix_title_artifacts` | repair what `.title()` broke |
| 7 | `expand_uk` | `"Uk"` -> `"United Kingdom"` |
| 8 | `expand_bare_cities` | `"Chicago"` -> `"Chicago, Illinois"` |
| 9 | `apply_aliases` | typos, translations, accent duplicates |

**Step 4 before step 5.** The Turkish dotted capital İ (U+0130) does not
survive `str.title()` — Python decomposes it into `I` plus a combining dot,
which then defeats every later string comparison and silently splits
`İstanbul` into two separate cities. Normalising it *first* is the only
reliable fix.

**Steps 8 and 9 after step 5.** Both are dictionary lookups. Their keys are
written in title case, so casing has to be settled before they run or nothing
matches.

In [3]:
from bandcamp_aotd.transform import locations as loc

examples = [
    "Osaka , Japan",
    "LONDON, UK",
    "Washington, D.C., D.C.",
    "İstanbul, Turkey",
    "Chicago",
    "Bogota, Colombia",
    "Kzn, South Africa",
]
pd.DataFrame({
    "raw": examples,
    "cleaned": [loc.clean_location(x) for x in examples],
})

,raw,cleaned
0,"Osaka , Japan","Osaka, Japan"
1,"LONDON, UK","London, United Kingdom"
2,"Washington, D.C., D.C.","Washington, D.C."
3,"İstanbul, Turkey","İstanbul, Turkey"
4,Chicago,"Chicago, Illinois"
5,"Bogota, Colombia","Bogotá, Colombia"
6,"Kzn, South Africa","KwaZulu-Natal, South Africa"


### Watching one value move through every step

Worth running when adding a new rule — it makes an ordering mistake obvious
immediately.

In [4]:
value = "  İSTANBUL ,  turkey  "
print(f"{'input':<24} {value!r}")
for step in loc.CLEANING_STEPS:
    value = step(value)
    print(f"{step.__name__:<24} {value!r}")

input                    '  İSTANBUL ,  turkey  '
fix_whitespace           'İSTANBUL, turkey'
fix_cjk                  'İSTANBUL, turkey'
fix_dc                   'İSTANBUL, turkey'
fix_turkish_dotted_i     'ISTANBUL, turkey'
fix_case                 'Istanbul, Turkey'
fix_title_artifacts      'Istanbul, Turkey'
expand_uk                'Istanbul, Turkey'
expand_bare_cities       'Istanbul, Turkey'
apply_aliases            'İstanbul, Turkey'


## Splitting into city / state / country

A cleaned string still isn't mappable — a choropleth needs a country column.
`split_location` handles the shapes that actually occur in the data:

- Countries whose own name contains a comma (São Tomé and Príncipe) are
  matched **first**, before any comma splitting can tear them apart.
- A single token is resolved by lookup: country, US state, Canadian province,
  or an unqualified city.
- UK constituent countries (England, Scotland, Wales, Northern Ireland) are
  returned as the **country**, not as a region under "United Kingdom" — that's
  how they appear standalone in the source data and how mapping tools expect
  them.

Country names come from `pycountry`, plus a hand-maintained alias map. The
alias map isn't redundant: ISO 3166 stores Turkey as "Türkiye", Czech Republic
as "Czechia" and Ivory Coast as "Côte d'Ivoire", none of which is what a label
owner types or what a dashboard reader wants to see.

In [5]:
cases = [
    "Brooklyn, New York",
    "Berlin, Germany",
    "Toronto, Ontario",
    "England, United Kingdom",
    "São Tomé, São Tomé and Príncipe",
    "Japan",
]
pd.DataFrame(
    [loc.split_location(c) for c in cases],
    columns=["city", "state", "country"],
    index=cases,
)

,city,state,country
"Brooklyn, New York",New York,New York,United States
"Berlin, Germany",Berlin,NaN,Germany
"Toronto, Ontario",Toronto,Ontario,Canada
"England, United Kingdom",NaN,England,United Kingdom
"São Tomé, São Tomé and Príncipe",São Tomé,NaN,São Tomé and Príncipe
Japan,NaN,NaN,Japan


## Build the full analytics table

`build_analytics_table` runs every transformation stage in order: text
normalisation, calendar features, geography, the independent-artist flag, the
deterministic `article_id`, de-duplication, and finally the canonical column
set.

One row gets rejected here — an article with no publication date. The grain of
the warehouse is "one published article", so a row without a date isn't a fact,
it's a parse failure. It's dropped loudly rather than allowed to sit in the
dashboard as a null-dated feature.

In [6]:
from bandcamp_aotd.transform import build_analytics_table

analytics = build_analytics_table(df)
print(analytics.shape)
analytics.head(3)

14:07:57 | INFO    | bandcamp_aotd.transform        | Transforming 2288 rows


14:07:57 | WARNING | bandcamp_aotd.transform        | Rejected 1 row(s) with no publication date (unusable grain): [{'article_id': 'c13ae6f60eff76e2', 'title': ',', 'artist': nan}]


14:07:57 | INFO    | bandcamp_aotd.transform        | Analytics table ready: 2287 rows, 80 countries, 343 authors, 25 genre tags


(2287, 41)


,article_id,article_url,article_slug,published_date,year,quarter,month,month_name,iso_week,day_of_month,...,spotify_release_date_precision,spotify_album_type,spotify_total_tracks,spotify_image_url,spotify_upc,spotify_ean,spotify_copyright,spotify_artist_status,spotify_artist_image_url,spotify_artist_url
0,f8bb3e7108af5b49,<NA>,<NA>,2011-10-28,2011,4,10,October,43,28,...,day,album,9.0,https://i.scdn.co/image/ab67616d0000b273257def...,NaN,NaN,NaN,matched,https://i.scdn.co/image/8a012df7c68f41f801f1f9...,https://open.spotify.com/artist/0UQCSEnTVyI8gt...
1,6994f0121e72977f,<NA>,<NA>,2011-11-06,2011,4,11,November,44,6,...,day,album,11.0,https://i.scdn.co/image/ab67616d0000b273106f3f...,NaN,NaN,NaN,matched,https://i.scdn.co/image/b21c26b4f4c381b762492a...,https://open.spotify.com/artist/3qPpwz6S0CbgMz...
2,01bfd63e9dbec0f0,<NA>,<NA>,2011-11-14,2011,4,11,November,46,14,...,day,single,1.0,https://i.scdn.co/image/ab67616d0000b27371a985...,NaN,NaN,NaN,matched,https://i.scdn.co/image/ab6761610000e5eb2badbd...,https://open.spotify.com/artist/77FIfTpCa6CKR5...


## Verify the cleaning actually worked

Three checks worth running every time the rules change.

In [7]:
raw_unique = df["label_location_raw"].nunique()
clean_unique = analytics["location_clean"].nunique()
unresolved = (analytics["location_clean"].notna() & analytics["country"].isna()).sum()

print(f"Raw distinct locations   : {raw_unique}")
print(f"Cleaned distinct locations: {clean_unique}  ({raw_unique - clean_unique} duplicates collapsed)")
print(f"Distinct countries        : {analytics['country'].nunique()}")
print(f"Unresolved (no country)   : {unresolved}   <- must be 0")
print(f"Location coverage         : {analytics['country'].notna().mean():.1%}")

Raw distinct locations   : 422
Cleaned distinct locations: 407  (15 duplicates collapsed)
Distinct countries        : 80
Unresolved (no country)   : 0   <- must be 0
Location coverage         : 96.5%


### Any location that still can't be resolved

This should print nothing. When it doesn't, the value goes into `ALIASES` or
`CITY_EXPANSIONS` in `locations.py` and gets a test case in
`tests/test_locations.py`.

In [8]:
missing = analytics.loc[
    analytics["location_clean"].notna() & analytics["country"].isna(),
    "location_clean",
].value_counts()
print(missing if len(missing) else "All locations resolved to a country.")

All locations resolved to a country.


## How the table reaches the warehouse

This notebook rebuilds the archive from the original export to show the
cleaning at work. It deliberately **writes nothing**:
`data/processed/aotd_analytics.csv` has one owner, the pipeline.

```bash
python -m bandcamp_aotd backfill   # once: load the original export
python -m bandcamp_aotd refresh    # weekly: new articles, merged in on article_id
```

An earlier version of this notebook ended with `write_csv(analytics)`. That
replaced the pipeline's table with the rows rebuilt here, silently dropping
every article a refresh had added since. Instead, the cell below checks that
the two agree.

In [9]:
committed = pd.read_csv(config.ANALYTICS_CSV)
in_committed = analytics["article_id"].isin(committed["article_id"])
added_since = ~committed["article_id"].isin(analytics["article_id"])

print(f"Rebuilt here from the export : {len(analytics):,} articles")
print(f"Pipeline's analytics table   : {len(committed):,} articles, "
      f"latest {committed['published_date'].max()}")
print(f"Rebuilt rows found in it     : {in_committed.sum():,}")
print(f"Added by refreshes since     : {added_since.sum():,}")
analytics.loc[~in_committed, ["published_date", "artist", "album"]]

Rebuilt here from the export : 2,287 articles
Pipeline's analytics table   : 2,368 articles, latest 2026-09-24
Rebuilt rows found in it     : 2,286
Added by refreshes since     : 82


,published_date,artist,album
2268,2026-04-29,Charbel Harber,May a soft sun bless your sky while you wait f...


Any rebuilt row missing from the pipeline's table was corrected by a later
scrape. Bandcamp sometimes fixes a headline after publication; a corrected
artist name changes the `article_id`, so the refresh replaces the old row.

## Next

`04_exploratory_analysis.ipynb` looks at what's actually in the archive.